In [34]:
import xarray as xr
import pandas as pd
import logging
import os

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define the path to the GRIB file
fie_address = os.getcwd() + '/../data/data/reanalysis-era5-land-temp/'
file_name = fie_address + 'reanalysis-era5-land_2022_07.grib'

def inspect_grib_file(file_path):
    if not os.path.exists(file_path):
        logging.error(f"File not found: {file_path}")
        return None
    
    logging.info(f"Opening GRIB file: {file_path}")
    try:
        with xr.open_dataset(file_path, engine='cfgrib') as ds:
            print("--- GRIB File Metadata ---")
            print(f"Dimensions: {ds.dims}")
            print(f"Coordinates: {list(ds.coords.keys())}")
            print(f"Data variables: {list(ds.data_vars.keys())}")
            
            for var in ds.data_vars:
                print(f"\nVariable '{var}':")
                print(f"  Attributes: {ds[var].attrs}")
            
            print("--------------------------")
            return ds
    except Exception as e:
        logging.error(f"Failed to open or inspect GRIB file: {e}")
        return None
ds = inspect_grib_file(file_name)

2025-08-13 14:54:26,061 - INFO - Opening GRIB file: /Users/abhisheksingh/work_new/climate_variable/notebooks/../data/data/reanalysis-era5-land-temp/reanalysis-era5-land_2022_07.grib
2025-08-13 14:54:26,064 - WARNING - Ignoring index file '/Users/abhisheksingh/work_new/climate_variable/notebooks/../data/data/reanalysis-era5-land-temp/reanalysis-era5-land_2022_07.grib.5b7b6.idx' incompatible with GRIB file


--- GRIB File Metadata ---
Dimensions: FrozenMappingWarningOnValuesAccess({'time': 32, 'step': 24, 'latitude': 42, 'longitude': 53})
Coordinates: ['number', 'time', 'step', 'surface', 'latitude', 'longitude', 'valid_time']
Data variables: ['t2m']

Variable 't2m':
  Attributes: {'GRIB_paramId': 167, 'GRIB_dataType': 'fc', 'GRIB_numberOfPoints': 2226, 'GRIB_typeOfLevel': 'surface', 'GRIB_stepUnits': 1, 'GRIB_stepType': 'instant', 'GRIB_gridType': 'regular_ll', 'GRIB_uvRelativeToGrid': 0, 'GRIB_NV': 0, 'GRIB_Nx': 53, 'GRIB_Ny': 42, 'GRIB_cfName': 'unknown', 'GRIB_cfVarName': 't2m', 'GRIB_gridDefinitionDescription': 'Latitude/Longitude Grid', 'GRIB_iDirectionIncrementInDegrees': 0.1, 'GRIB_iScansNegatively': 0, 'GRIB_jDirectionIncrementInDegrees': 0.1, 'GRIB_jPointsAreConsecutive': 0, 'GRIB_jScansPositively': 0, 'GRIB_latitudeOfFirstGridPointInDegrees': 36.25, 'GRIB_latitudeOfLastGridPointInDegrees': 32.15, 'GRIB_longitudeOfFirstGridPointInDegrees': 75.5, 'GRIB_longitudeOfLastGridPointInDe

/Users/abhisheksingh/work_new/climate_variable/.venv/lib/python3.13/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


In [35]:
df = ds.to_dataframe().reset_index()
print("Successfully converted to DataFrame.")
df['time'] = pd.to_datetime(df['time'])

df.set_index('time', inplace=True)
df['t2m_celsius'] = df['t2m'] - 273.15



Successfully converted to DataFrame.


In [29]:
df


,step,latitude,longitude,number,surface,valid_time,t2m,t2m_celsius
time,,,,,,,,
2021-12-31,0 days 01:00:00,36.25,75.5,0,0.0,2021-12-31 01:00:00,NaN,NaN
2021-12-31,0 days 01:00:00,36.25,75.6,0,0.0,2021-12-31 01:00:00,NaN,NaN
2021-12-31,0 days 01:00:00,36.25,75.7,0,0.0,2021-12-31 01:00:00,NaN,NaN
2021-12-31,0 days 01:00:00,36.25,75.8,0,0.0,2021-12-31 01:00:00,NaN,NaN
2021-12-31,0 days 01:00:00,36.25,75.9,0,0.0,2021-12-31 01:00:00,NaN,NaN
...,...,...,...,...,...,...,...,...
2022-01-31,1 days 00:00:00,32.15,80.3,0,0.0,2022-02-01 00:00:00,NaN,NaN
2022-01-31,1 days 00:00:00,32.15,80.4,0,0.0,2022-02-01 00:00:00,NaN,NaN
2022-01-31,1 days 00:00:00,32.15,80.5,0,0.0,2022-02-01 00:00:00,NaN,NaN


In [53]:
df.shape


(1709568, 8)

In [54]:
df['t2m_celsius'].resample('M').min() 
# min_monthly_temp

/var/folders/f7/s6ns0c0171z2_qrs6rhh6w580000gn/T/ipykernel_42430/1444629392.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df['t2m_celsius'].resample('M').min()


time
2022-06-30   -14.071869
2022-07-31   -19.921768
Freq: ME, Name: t2m_celsius, dtype: float32

In [41]:
min_monthly_temp

time
2022-06-30   -14.071869
2022-07-31   -19.921768
Freq: ME, Name: t2m_celsius, dtype: float32